In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from tqdm import tqdm
import torch
import torch.optim as optim
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
import torchvision.models as models
import torchaudio
import torchaudio.transforms as AT

# ==========================================
# 1. 基础配置与路径 
# ==========================================
wav_sec = 5
sample_rate = 32000
min_segment = sample_rate * wav_sec
epochs = 8
batch_size = 32
learning_rate = 0.001

POSSIBLE_ROOTS = [
    '/kaggle/input/competitions/birdclef-2026/',
]
ROOT_PATH = next((path for path in POSSIBLE_ROOTS if os.path.exists(path)), POSSIBLE_ROOTS[0])
TRAIN_AUDIO_PATH = os.path.join(ROOT_PATH, 'train_audio/')

# 获取 234 个完整的比赛类别
taxonomy = pd.read_csv(os.path.join(ROOT_PATH, 'taxonomy.csv'))
CLASS_LABELS = sorted(taxonomy['primary_label'].unique().tolist())
num_classes = len(CLASS_LABELS)

# 读取训练元数据
train_meta = pd.read_csv(os.path.join(ROOT_PATH, 'train.csv'))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device} | Total Classes: {num_classes}")

# ==========================================
# 2. 特征提取配置
# ==========================================
mel_spectrogram = AT.MelSpectrogram(
    sample_rate=sample_rate,
    n_fft=1024,
    win_length=1024,
    hop_length=512,
    center=True,
    f_min=20,
    f_max=15000,
    pad_mode="reflect",
    power=2.0,
    norm='slaney',
    n_mels=128,
    mel_scale="htk"
)

# ==========================================
# 3. 数据集构建 (BirdclefDataset)
# ==========================================
class BirdclefDataset(Dataset):
    def __init__(self, df, mode='train'):
        self.df = df
        self.mode = mode

    def normalize_std(self, spec, eps=1e-23):
        mean = torch.mean(spec)
        std = torch.std(spec)
        return torch.where(std == 0, spec - mean, (spec - mean) / (std + eps))
                
    def __getitem__(self, index):
        row = self.df.iloc[index]
        file_path = os.path.join(TRAIN_AUDIO_PATH, row.filename)
        
        # 加载音频并转单声道
        sig, sr = torchaudio.load(file_path, backend="soundfile")
        if sig.shape[0] > 1: 
            sig = sig.mean(0, keepdim=True)
            
        # 长度不足时补齐，超长时随机截取 (你的经典增强方法)
        if sig.shape[1] < min_segment:
            sig = torch.cat([sig, torch.zeros(1, min_segment - sig.shape[1])], dim=1)
        elif sig.shape[1] > min_segment:
            max_start = sig.shape[1] - min_segment
            start = np.random.randint(0, max_start)
            sig = sig[:, start : start + min_segment]
            
        # 归一化与微弱噪声注入
        sig = sig / (torch.max(torch.abs(sig)) + 1e-7)
        sig = sig + 1.5849e-05 * (torch.rand(1, min_segment) - 0.5) 
        
        # 转换为 Mel 频谱
        melspec = mel_spectrogram(sig)
        melspec = torch.log(melspec + 1e-10)
        melspec = self.normalize_std(melspec)

        # 生成 234 维的 One-Hot 标签
        target = row.primary_label
        y = np.array([1.0 if item == target else 0.0 for item in CLASS_LABELS], dtype=np.float32)
        
        return melspec, torch.tensor(y)
    
    def __len__(self):
        return len(self.df)

# 划分训练集与验证集
# ==========================================
# 优化版：处理极端不平衡数据的训练验证划分
# ==========================================

# 1. 统计每个类别的样本数
class_counts = train_meta['primary_label'].value_counts()

# 2. 找出只有 1 个样本的“极度罕见”类别
rare_classes = class_counts[class_counts < 2].index

# 3. 将数据集拆分为“极度罕见”和“可分层”两部分
rare_df = train_meta[train_meta['primary_label'].isin(rare_classes)]
common_df = train_meta[~train_meta['primary_label'].isin(rare_classes)]

# 4. 仅对拥有 >= 2 个样本的常见类别进行正常的分层划分
train_common, val_df = train_test_split(
    common_df, 
    test_size=0.2, 
    random_state=42, 
    stratify=common_df['primary_label']
)

# 5. 将极度罕见的样本强行加入训练集（保证模型至少能学到一次！）
train_df = pd.concat([train_common, rare_df]).sample(frac=1.0, random_state=42).reset_index(drop=True)
val_df = val_df.reset_index(drop=True)

print(f"Total Train Samples: {len(train_df)} | Total Val Samples: {len(val_df)}")
print(f"Forced {len(rare_df)} rare single-sample classes directly into training set.")


train_dataset = BirdclefDataset(train_df, mode='train')
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2, drop_last=True)

val_dataset = BirdclefDataset(val_df, mode='val')
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=2, drop_last=False)

# ==========================================
# 4. 模型定义 (ResNet34)
# ==========================================
class Model_resnet34(nn.Module):
    def __init__(self, num_classes, pretrained=True):
        super().__init__()
        weights = models.ResNet34_Weights.DEFAULT if pretrained else None
        self.model = models.resnet34(weights=weights)
        num_ftrs = self.model.fc.in_features
        self.model.fc = nn.Linear(num_ftrs, num_classes)

    def forward(self, x):
        # 将单通道频谱复制为三通道输入
        x = torch.cat((x, x, x), 1)
        return self.model(x)

model = Model_resnet34(num_classes=num_classes, pretrained=True).to(device)

# ==========================================
# 5. 训练循环 (带进度条)
# ==========================================
# 注意：针对多标签/One-Hot，BCEWithLogitsLoss 比 CrossEntropy 更稳定
criterion = nn.BCEWithLogitsLoss() 
optimizer = optim.Adam(model.parameters(), lr=learning_rate)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

best_val_loss = float('inf')

for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs} [Train]")
    for melspecs, labels in pbar:
        melspecs, labels = melspecs.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(melspecs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        pbar.set_postfix({'loss': f"{loss.item():.4f}"})
        
    avg_train_loss = running_loss / len(train_loader)
    
    # 验证环节
    model.eval()
    running_loss_val = 0.0
    with torch.no_grad():
        val_pbar = tqdm(val_loader, desc=f"Epoch {epoch+1}/{epochs} [Val  ]")
        for melspecs, labels in val_pbar:
            melspecs, labels = melspecs.to(device), labels.to(device)
            outputs = model(melspecs)
            loss = criterion(outputs, labels)
            running_loss_val += loss.item()
            val_pbar.set_postfix({'val_loss': f"{loss.item():.4f}"})
            
    avg_val_loss = running_loss_val / len(val_loader)
    scheduler.step()
    
    print(f"Epoch {epoch+1} Summary -> Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}")
    
    # 保存最佳模型
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        torch.save(model.state_dict(), "baseline_2026.pth")
        print(f"--> Saved better model with Val Loss: {best_val_loss:.4f}")

print("Training Complete!")

Using device: cuda | Total Classes: 234
Total Train Samples: 28440 | Total Val Samples: 7109
Forced 4 rare single-sample classes directly into training set.
Downloading: "https://download.pytorch.org/models/resnet34-b627a593.pth" to /root/.cache/torch/hub/checkpoints/resnet34-b627a593.pth


100%|██████████| 83.3M/83.3M [00:00<00:00, 179MB/s]
Epoch 1/8 [Train]:   0%|          | 0/888 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torchaudio/__init__.py:86: UserWarning: The 'backend' parameter is not used by TorchCodec AudioDecoder.
  return load_with_torchcodec(
/usr/local/lib/python3.12/dist-packages/torchaudio/__init__.py:86: UserWarning: The 'backend' parameter is not used by TorchCodec AudioDecoder.
  return load_with_torchcodec(
Epoch 1/8 [Val  ]:   0%|          | 0/223 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torchaudio/__init__.py:86: UserWarning: The 'backend' parameter is not used by TorchCodec AudioDecoder.
  return load_with_torchcodec(
/usr/local/lib/python3.12/dist-packages/torchaudio/__init__.py:86: UserWarning: The 'backend' parameter is not used by TorchCodec AudioDecoder.
  return load_with_torchcodec(
Epoch 1/8 [Val  ]: 100%|██████████| 223/223 [02:44<00:00,  1.36it/s, val_loss=0.0134]


Epoch 1 Summary -> Train Loss: 0.0248 | Val Loss: 0.0190
--> Saved better model with Val Loss: 0.0190


Epoch 2/8 [Train]:   0%|          | 0/888 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torchaudio/__init__.py:86: UserWarning: The 'backend' parameter is not used by TorchCodec AudioDecoder.
  return load_with_torchcodec(
/usr/local/lib/python3.12/dist-packages/torchaudio/__init__.py:86: UserWarning: The 'backend' parameter is not used by TorchCodec AudioDecoder.
  return load_with_torchcodec(
Epoch 2/8 [Val  ]:   0%|          | 0/223 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torchaudio/__init__.py:86: UserWarning: The 'backend' parameter is not used by TorchCodec AudioDecoder.
  return load_with_torchcodec(
/usr/local/lib/python3.12/dist-packages/torchaudio/__init__.py:86: UserWarning: The 'backend' parameter is not used by TorchCodec AudioDecoder.
  return load_with_torchcodec(
Epoch 2/8 [Val  ]: 100%|██████████| 223/223 [02:14<00:00,  1.66it/s, val_loss=0.0203]


Epoch 2 Summary -> Train Loss: 0.0164 | Val Loss: 0.0152
--> Saved better model with Val Loss: 0.0152


...

Epoch 6 Summary -> Train Loss: 0.0086 | Val Loss: 0.0097
--> Saved better model with Val Loss: 0.0097


Epoch 7/8 [Train]:   0%|          | 0/888 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torchaudio/__init__.py:86: UserWarning: The 'backend' parameter is not used by TorchCodec AudioDecoder.
  return load_with_torchcodec(
/usr/local/lib/python3.12/dist-packages/torchaudio/__init__.py:86: UserWarning: The 'backend' parameter is not used by TorchCodec AudioDecoder.
  return load_with_torchcodec(
Epoch 7/8 [Val  ]:   0%|          | 0/223 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torchaudio/__init__.py:86: UserWarning: The 'backend' parameter is not used by TorchCodec AudioDecoder.
  return load_with_torchcodec(
/usr/local/lib/python3.12/dist-packages/torchaudio/__init__.py:86: UserWarning: The 'backend' parameter is not used by TorchCodec AudioDecoder.
  return load_with_torchcodec(
Epoch 7/8 [Val  ]: 100%|██████████| 223/223 [02:15<00:00,  1.65it/s, val_loss=0.0022]


Epoch 7 Summary -> Train Loss: 0.0075 | Val Loss: 0.0090
--> Saved better model with Val Loss: 0.0090


Epoch 8/8 [Train]:   0%|          | 0/888 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torchaudio/__init__.py:86: UserWarning: The 'backend' parameter is not used by TorchCodec AudioDecoder.
  return load_with_torchcodec(
/usr/local/lib/python3.12/dist-packages/torchaudio/__init__.py:86: UserWarning: The 'backend' parameter is not used by TorchCodec AudioDecoder.
  return load_with_torchcodec(
Epoch 8/8 [Val  ]:   0%|          | 0/223 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torchaudio/__init__.py:86: UserWarning: The 'backend' parameter is not used by TorchCodec AudioDecoder.
  return load_with_torchcodec(
/usr/local/lib/python3.12/dist-packages/torchaudio/__init__.py:86: UserWarning: The 'backend' parameter is not used by TorchCodec AudioDecoder.
  return load_with_torchcodec(
Epoch 8/8 [Val  ]: 100%|██████████| 223/223 [02:15<00:00,  1.64it/s, val_loss=0.0083]


Epoch 8 Summary -> Train Loss: 0.0068 | Val Loss: 0.0087
--> Saved better model with Val Loss: 0.0087
Training Complete!
